# RMSNorm

In [3]:
import torch
import torch.nn.functional as F
from torch import nn

In [4]:
class Qwen3RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        # eps是为了防止除以0的情况
        self.eps = eps
        # weight是一个可学习的参数，全部初始化为1
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        # 计算RMSNorm的核心部分
        # x.pow(2).mean(-1, keepdim=True)计算了输入x的平方的均值
        # torch.rsqrt是平方根的倒数，这样就得到了RMSNorm的分母部分，再加上eps防止分母为0
        # 最后乘以x，得到RMSNorm的结果
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # forward函数是模型的前向传播
        # 首先将输入x转为float类型，然后进行RMSNorm，最后再转回原来的数据类型
        # 最后乘以weight，这是RMSNorm的一个可学习的缩放因子
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

rms_norm = Qwen3RMSNorm(dim=4, eps=1e-6)

In [5]:
import torch

# 创建输入张量
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0],
                   [2.0, 4.0, 6.0, 8.0],
                   [3.0, 6.0, 9.0, 12.0]],
                  
                  [[-1.0, -2.0, -3.0, -4.0],
                   [-2.0, -4.0, -6.0, -8.0],
                   [-3.0, -6.0, -9.0, -12.0]]])

print("输入张量 x 的形状:", x.shape)  # torch.Size([2, 3, 4])
print("x = \n", x)

输入张量 x 的形状: torch.Size([2, 3, 4])
x = 
 tensor([[[  1.,   2.,   3.,   4.],
         [  2.,   4.,   6.,   8.],
         [  3.,   6.,   9.,  12.]],

        [[ -1.,  -2.,  -3.,  -4.],
         [ -2.,  -4.,  -6.,  -8.],
         [ -3.,  -6.,  -9., -12.]]])


In [7]:
# 计算整个批次的RMSNorm
output = rms_norm(x)
print("\nRMSNorm输出形状:", output.shape)  # torch.Size([2, 3, 4])
print("输出第一个样本的第一个位置:", output[0, 0])

# 验证手动计算
# 手动计算第一个样本的第一个位置
x_sample = x[0, 0].float()
rms = torch.sqrt(x_sample.pow(2).mean() + 1e-6)
scale = 1 / rms
manual_result = x_sample * scale * rms_norm.weight

print("\n手动计算结果:", manual_result)
print("代码计算结果:", output)
print("两者是否接近:", torch.allclose(manual_result, output[0, 0], rtol=1e-4))


RMSNorm输出形状: torch.Size([2, 3, 4])
输出第一个样本的第一个位置: tensor([0.3651, 0.7303, 1.0954, 1.4606], grad_fn=<SelectBackward0>)

手动计算结果: tensor([0.3651, 0.7303, 1.0954, 1.4606], grad_fn=<MulBackward0>)
代码计算结果: tensor([[[ 0.3651,  0.7303,  1.0954,  1.4606],
         [ 0.3651,  0.7303,  1.0954,  1.4606],
         [ 0.3651,  0.7303,  1.0954,  1.4606]],

        [[-0.3651, -0.7303, -1.0954, -1.4606],
         [-0.3651, -0.7303, -1.0954, -1.4606],
         [-0.3651, -0.7303, -1.0954, -1.4606]]], grad_fn=<MulBackward0>)
两者是否接近: True


# Attention

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- 1. 基础配置类 ---
class Qwen3Config:
    def __init__(self):
        self.hidden_size = 1024
        self.num_attention_heads = 16
        self.num_key_value_heads = 8  # 使用 GQA (Grouped Query Attention)
        self.head_dim = 128
        self.attention_dropout = 0.0
        self.attention_bias = False
        self.rms_norm_eps = 1e-6
        self.layer_types = ["sliding_attention"] * 12
        self.sliding_window = 1024
        self._attn_implementation = "eager"
        self.vocab_size = 10000
        self.num_hidden_layers = 8

# --- 2. 归一化层 ---
class Qwen3RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # 针对最后一位 head_dim 进行 norm
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x

# --- 3. 旋转位置编码 (RoPE) 简化版 ---
def apply_rotary_pos_emb(q, k, cos, sin):
    # 这里的实现为了简单，假设 cos/sin 已经对齐了维度
    # q, k: [batch, heads, seq_len, head_dim]
    def rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

# --- 4. 核心 Attention 类 ---
class Qwen3Attention(nn.Module):
    def __init__(self, config: Qwen3Config, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = config.head_dim
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.scaling = self.head_dim**-0.5

        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=config.attention_bias)
        
        # Qwen3 特色：在投影后对 QK 进行 Head-wise RMSNorm
        self.q_norm = Qwen3RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = Qwen3RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(self, hidden_states, cos, sin, attention_mask=None):
        bsz, q_len, _ = hidden_states.size()

        # 1. 线性投影并 Reshape 为 [batch, seq, heads, head_dim]
        query_states = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim)
        key_states = self.k_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim)
        value_states = self.v_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim)

        # 2. QK Norm (Qwen3 的关键步骤)
        query_states = self.q_norm(query_states).transpose(1, 2)
        key_states = self.k_norm(key_states).transpose(1, 2)
        value_states = value_states.transpose(1, 2)

        print(f'key_states shape: {key_states.shape}')
        
        # 3. 应用 RoPE
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        # 4. GQA: 复制 K/V 头部以匹配 Q 的头部数量
        key_states = torch.repeat_interleave(key_states, dim=1, repeats=self.num_kv_groups)
        value_states = torch.repeat_interleave(value_states, dim=1, repeats=self.num_kv_groups)

        print(f'key_states shape: {key_states.shape}')

        # 5. 标准 Scaled Dot-Product Attention
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) * self.scaling
        
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask

        attn_weights = F.softmax(attn_weights, dim=-1).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, value_states)

        # 6. 合并头部并输出
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, -1)
        return self.o_proj(attn_output), attn_weights

In [21]:
# --- 5. 测试运行 ---
def run_example():
    config = Qwen3Config()
    model = Qwen3Attention(config, layer_idx=0)
    
    # 模拟输入参数
    batch_size = 64
    seq_len = 256
    hidden_dim = config.hidden_size
    head_dim = config.head_dim
    
    # 1. 随机生成隐藏状态
    hidden_states = torch.randn(batch_size, seq_len, hidden_dim)
    
    # 2. 构造简易 RoPE cos/sin (实际应用中由专用类生成)
    cos = torch.ones(1, 1, seq_len, head_dim)
    sin = torch.zeros(1, 1, seq_len, head_dim)
    
    # 3. 构造因果掩码 (Causal Mask)
    mask = torch.full((seq_len, seq_len), float("-inf"))
    mask = torch.triu(mask, diagonal=1)
    
    # 运行
    output, weights = model(hidden_states, cos, sin, attention_mask=mask)
    
    print(f"输入形状: {hidden_states.shape}")
    print(f"输出形状: {output.shape}")
    print(f"注意力权重形状: {weights.shape}")


run_example()

key_states shape: torch.Size([64, 8, 256, 128])
key_states shape: torch.Size([64, 16, 256, 128])
输入形状: torch.Size([64, 256, 1024])
输出形状: torch.Size([64, 256, 1024])
注意力权重形状: torch.Size([64, 16, 256, 256])


In [ ]:
from transformers import Qwen3Model

model = Qwen3Model.from_pretrained("Qwen/Qwen3-72B-Instruct")

model.generate("你好")

# Decoder

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. MLP 层实现 (SwiGLU) ---
class Qwen3MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Qwen 通常会将中间层维度扩大
        self.intermediate_size = getattr(config, "intermediate_size", config.hidden_size * 4)
        self.gate_proj = nn.Linear(config.hidden_size, self.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, self.intermediate_size, bias=False)
        self.down_proj = nn.Linear(self.intermediate_size, config.hidden_size, bias=False)
        self.act_fn = F.silu # SiLU 即 Swish

    def forward(self, x):
        # SwiGLU: (SiLU(gate) * up) -> down
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

# --- 2. 简化的 Decoder 层 ---
class Qwen3DecoderLayer(nn.Module):
    def __init__(self, config, layer_idx: int):
        super().__init__()
        self.hidden_size = config.hidden_size

        # 引用之前定义的 Attention 层
        self.self_attn = Qwen3Attention(config=config, layer_idx=layer_idx)
        self.mlp = Qwen3MLP(config)
        
        self.input_layernorm = Qwen3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = Qwen3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_embeddings=None, # (cos, sin)
    ):
        # --- Pre-Norm 结构 ---
        # 1. Self Attention 路径
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        
        hidden_states, attn_weights = self.self_attn(
            hidden_states=hidden_states,
            cos=position_embeddings[0],
            sin=position_embeddings[1],
            attention_mask=attention_mask,
        )
        hidden_states = residual + hidden_states

        # 2. MLP 路径
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        return hidden_states, attn_weights

In [11]:
def run_decoder_example():
    # 1. 基础配置

    config = Qwen3Config()
    layer = Qwen3DecoderLayer(config, layer_idx=0)
    
    # 2. 模拟输入数据
    batch_size = 1
    seq_len = 8
    hidden_states = torch.randn(batch_size, seq_len, config.hidden_size)

    # 3. 构造 RoPE (cos, sin)
    # 实际模型中这些是从 RoPE Embedding 类生成的，这里手动构造对应维度的 Tensor
    cos = torch.ones(batch_size, 1, seq_len, config.head_dim)
    sin = torch.zeros(batch_size, 1, seq_len, config.head_dim)
    position_embeddings = (cos, sin)

    # 4. 前向传播
    with torch.no_grad():
        output, weights = layer(
            hidden_states=hidden_states,
            position_embeddings=position_embeddings
        )

    print(f"Decoder Layer 输入形状: {hidden_states.shape}")
    print(f"Decoder Layer 输出形状: {output.shape}")
    print("--- 运行成功 ---")


run_decoder_example()

Decoder Layer 输入形状: torch.Size([1, 8, 1024])
Decoder Layer 输出形状: torch.Size([1, 8, 1024])
--- 运行成功 ---


In [15]:
import torch
import torch.nn as nn

# --- 1. RoPE 旋转位置嵌入生成器 ---
class Qwen3RotaryEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dim = config.head_dim
        self.base = 10000
        # 计算频率因子
        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2).float() / self.dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, position_ids):
        # position_ids: [batch, seq_len]
        inv_freq_expansion = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)
        position_ids_expansion = position_ids[:, None, :].float()
        # 外积计算矩阵
        freqs = (inv_freq_expansion @ position_ids_expansion).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()

# --- 2. 简易因果掩码生成 ---
def create_causal_mask(seq_len, device):
    mask = torch.full((seq_len, seq_len), float("-inf"), device=device)
    mask = torch.triu(mask, diagonal=1)
    return mask # [seq_len, seq_len]

In [16]:
class Qwen3Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList(
            [Qwen3DecoderLayer(config, i) for i in range(config.num_hidden_layers)]
        )
        self.norm = Qwen3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = Qwen3RotaryEmbedding(config)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        
        # 1. Token Embedding
        hidden_states = self.embed_tokens(input_ids)
        
        # 2. 准备位置信息和掩码
        position_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).repeat(batch_size, 1)
        cos, sin = self.rotary_emb(hidden_states, position_ids)
        position_embeddings = (cos.unsqueeze(1), sin.unsqueeze(1)) # 适配 Attention 的 unsqueeze_dim=1
        
        mask = create_causal_mask(seq_len, input_ids.device)

        # 3. 逐层通过 Decoder
        for layer in self.layers:
            # 简化调用，只取第一个返回值 hidden_states
            hidden_states, _ = layer(
                hidden_states,
                attention_mask=mask,
                position_embeddings=position_embeddings
            )

        # 4. 最后的归一化
        hidden_states = self.norm(hidden_states)
        return hidden_states

In [17]:
def test_full_model():
    config  = Qwen3Config()
    model = Qwen3Model(config)
    
    # 模拟输入 ID (Batch=2, Seq=10)
    input_ids = torch.randint(0, config.vocab_size, (2, 10))
    
    # 前向传播
    with torch.no_grad():
        last_hidden_states = model(input_ids)
        
    print(f"输入形状: {input_ids.shape}")
    print(f"最终输出形状 (Last Hidden State): {last_hidden_states.shape}")
    print("--- 模型全流程跑通 ---")

test_full_model()

输入形状: torch.Size([2, 10])
最终输出形状 (Last Hidden State): torch.Size([2, 10, 1024])
--- 模型全流程跑通 ---
